# Mini-TP 4 — Puntúa tu modelo sobre un flujo

Operaciones de Aprendizaje Automático II · CEIA – FIUBA · Entrega individual

Aplica el mismo modelo que los TP1, TP2 y TP3 —XGBoost que predice si un crimen reportado en
Chicago termina en arresto— sobre un flujo de eventos, en lugar de pedidos sueltos.

## Qué se entrega

1. Un flujo de eventos con los reportes que consume el modelo.
2. Un consumidor que puntúa cada evento apenas llega.
3. Métricas por ventana: throughput, latencia p95 y un indicador de drift.
4. Una alerta al cruzar el umbral de drift.
5. La comparación contra puntuar el mismo lote en batch, y la reflexión.

Como en los TPs anteriores, los `.py` de `tp4_streaming/` son la fuente y están cubiertos por
tests; el notebook los importa y los muestra.

In [1]:
import logging
import os
from pathlib import Path

import arrest_model

# El notebook vive en tp4_streaming/, pero los imports esperan la raíz del repo.
REPO = Path(arrest_model.__file__).resolve().parent.parent
os.chdir(REPO)

# kafka-python habla mucho por el log y acá taparía la salida del flujo.
logging.getLogger("kafka").setLevel(logging.WARNING)
print("listo, desde", REPO)

listo, desde /Users/belcattaneo/Documents/MIA/MLOPS2/mlops2-minitps


## 1. El modelo

Sale del mismo `model/model.pkl` que sirven las tres APIs del repo. El flujo no reimplementa
nada: valida con el mismo contrato, codifica con las mismas funciones y predice con el mismo
modelo.

In [2]:
from arrest_model.model import load_bundle

bundle = load_bundle()
metadata = bundle["metadata"]
print(metadata["name"], "v", metadata["version"], "·", metadata["framework"])
print("features:", ", ".join(metadata["features"]))

chicago-arrest-xgboost v 1 · xgboost 3.4.1
features: IUCR_freq, Primary_Type_freq, Location_Description_freq, Day_sin, X Coordinate_standardized, Y Coordinate_standardized, Distance Crime To Police Station_standardized


## 2. El flujo

Los eventos son reportes sintéticos generados alrededor de la zona donde el modelo vio la
mayoría de los crímenes. A mitad del flujo se corren 13 km al norte: ese es el cambio de
distribución que después tiene que detectar el drift.

In [3]:
from tp4_streaming.events import crime_stream

for evento in list(crime_stream(2, seed=9)):
    print(evento)

{'iucr': '0820', 'primary_type': 'CRIMINAL DAMAGE', 'location_description': 'APARTMENT', 'date': '2024-12-31T23:58:00', 'latitude': 41.837535, 'longitude': -87.666706}
{'iucr': '1310', 'primary_type': 'HOMICIDE', 'location_description': 'APARTMENT', 'date': '2024-12-31T23:59:00', 'latitude': 41.841061, 'longitude': -87.659445}


## 3. El consumidor: inferencia online y métricas por ventana

Cada evento se valida, se codifica y se puntúa apenas llega. La ventana es deslizante: guarda lo
que llegó en el último segundo y descarta lo viejo, así las métricas describen cómo viene el
flujo ahora.

El indicador de drift son las tres features estandarizadas del modelo. El `.pkl` guarda la media
y el desvío con los que se entrenó, así que en entrenamiento valen 0 y 1: cuánto se aleja de 0 la
media de la ventana es cuánto se corrió la entrada, medido en desvíos. La alerta salta cuando
cruza 0.5, y no se repite mientras el drift dure.

In [4]:
from IPython.display import Code, display

display(Code(filename="tp4_streaming/consumer.py", language="python"))

"""Mini-TP 4: el consumidor que puntúa el flujo evento por evento.

Cada evento se valida con el mismo contrato que la API REST, se codifica y se puntúa apenas
llega: eso es inferencia online. Cada tantos eventos se cierra una ventana y se informa cómo
viene el flujo, que es lo que se monitorea en producción.
"""

import time
from collections import deque
from collections.abc import Callable, Iterable, Iterator
from dataclasses import dataclass
from statistics import mean
from typing import Any

from arrest_model.model import predict
from arrest_model.schemas import CrimeReport
from tp4_streaming.metrics import drift_indicator, p95, window_throughput

WINDOW_SECONDS = 10.0
REPORT_EVERY = 50
# Medio desvío de corrimiento sostenido respecto del entrenamiento ya es un cambio real y no
# ruido del flujo, que se mueve alrededor de 0.3.
DRIFT_THRESHOLD = 0.5


@dataclass(frozen=True)
class WindowReport:
    """Cómo venía el flujo en una ventana."""

    scored: int  # eventos puntuados desde que arrancó el flujo
    events: int  # eventos dentro de la ventana
    throughput: float  # eventos por segundo
    p95_ms: float  # latencia que el 95% de los eventos de la ventana no supera
    drift: float  # desvíos de distancia entre las entradas de la ventana y el entrenamiento
    mean_probability: float  # probabilidad media de arresto que predijo el modelo
    alert: bool  # True solo en la ventana en la que el drift cruza el umbral


def score_stream(
    source: Iterable[dict[str, Any]],
    bundle: dict[str, Any],
    *,
    window_seconds: float = WINDOW_SECONDS,
    report_every: int = REPORT_EVERY,
    threshold: float = DRIFT_THRESHOLD,
    clock: Callable[[], float] = time.monotonic,
) -> Iterator[WindowReport]:
    """Puntúa cada evento del flujo y cierra una ventana cada `report_every` eventos.

    La ventana es deslizante: guarda lo que llegó en los últimos `window_seconds` y descarta lo
    viejo, así las métricas describen cómo viene el flujo ahora y no desde que arrancó. El corte
    es semiabierto, así que con un evento por segundo una ventana de 10 segundos tiene 10
    eventos y no 11.

    La alerta salta en la ventana en la que el drift cruza el umbral, no en todas las que siguen:
    mientras el drift dure, repetir el aviso en cada ventana sería ruido.

    El reloj entra por parámetro para que los tests fijen los tiempos y las métricas no dependan
    de cuánto tarde la máquina.
    """
    window: deque[tuple[float, CrimeReport, float, float]] = deque()
    scored = 0
    drifting = False
    for event in source:
        report = CrimeReport.model_validate(event)
        started = time.perf_counter()
        prediction = predict(bundle, [report])[0]
        latency_ms = (time.perf_counter() - started) * 1000

        now = clock()
        window.append((now, report, latency_ms, prediction.probability))
        while window and now - window[0][0] >= window_seconds:
            window.popleft()

        scored += 1
        if scored % report_every == 0:
            elapsed = window[-1][0] - window[0][0]
            drift = drift_indicator([item[1] for item in window], bundle["params"])
            crossed = drift > threshold and not drifting
            drifting = drift > threshold
            yield WindowReport(
                scored=scored,
                events=len(window),
                throughput=window_throughput(len(window), elapsed),
                p95_ms=p95([item[2] for item in window]),
                drift=drift,
                mean_probability=mean(item[3] for item in window),
                alert=crossed,
            )

In [5]:
from tp4_streaming.run import run_stream

run_stream(events=400, report_every=100, drift_from=200)

ventana @  100 eventos |  85 en ventana | throughput    85.1 ev/s | p95  6.48 ms | drift  0.23 | probabilidad media 0.489


ventana @  200 eventos |  82 en ventana | throughput    82.9 ev/s | p95  7.58 ms | drift  0.22 | probabilidad media 0.372


ventana @  300 eventos |  85 en ventana | throughput    85.8 ev/s | p95  7.47 ms | drift  1.38 | probabilidad media 0.377  ← ALERTA de drift


ventana @  400 eventos |  84 en ventana | throughput    85.1 ev/s | p95  7.92 ms | drift  1.37 | probabilidad media 0.510


0

Las dos primeras ventanas están tranquilas. Después del evento 200 los reportes llegan de otra
zona, el indicador salta por encima de 0.5 y suena la alerta. La última ventana sigue alta y no
vuelve a avisar: el aviso es del cruce, no de cada ventana.

Mirá también la probabilidad media: casi no se mueve. El modelo sigue contestando con toda
tranquilidad aunque le estén llegando datos de un lugar que casi no conoce, y por eso el
monitoreo mira lo que entra y no solo lo que sale.

## 4. Comparación contra batch

Los mismos eventos, puntuados de a uno y puntuados de una sola vez. Los dos caminos hacen el
mismo trabajo —validar, codificar y predecir— y se miden del mismo lado, sin la cola ni el ritmo
de llegada.

In [6]:
from tp4_streaming.compare import compare, format_comparison

print(format_comparison(compare(list(crime_stream(1000, seed=6)), bundle)))

Los mismos 1000 eventos, puntuados de las dos formas.

| camino | total (ms) | por evento (ms) |
|---|---|---|
| online | 1303.8 | 1.304 |
| batch | 5.0 | 0.005 |

Las predicciones coinciden: True.


### Reflexión

> Pendiente de completar: streaming contra batch, cómo se detectó el drift y qué hacer al
> dispararse la alerta.

Lo medido, para apoyarse:

- El batch puntúa cada evento unas 300 veces más rápido, porque el costo fijo de llamar al
  modelo se reparte entre todos los eventos del lote.
- El streaming puntúa el reporte apenas entra, mientras que el batch entrega recién cuando corre.
- Las predicciones de los dos caminos coinciden exactamente, así que la diferencia es de costo y
  de frescura, no de resultado.
- El drift se detecta sobre las entradas: la salida del modelo casi no se movió.

## 5. El mismo flujo contra Kafka

El consumidor no sabe de dónde vienen los eventos, así que corre igual contra un broker real. Con
Redpanda levantado (`make redpanda-up`), esta celda publica en el topic y consume desde ahí; sin
broker, lo informa y sigue.

In [7]:
import socket


def hay_broker() -> bool:
    with socket.socket() as sock:
        sock.settimeout(0.5)
        return sock.connect_ex(("127.0.0.1", 9092)) == 0


if hay_broker():
    run_stream(events=400, report_every=100, drift_from=200, source="kafka")
else:
    print("No hay broker en 9092. Se levanta con `make redpanda-up` y se corre `make stream-kafka`.")

ventana @  100 eventos |  69 en ventana | throughput    69.5 ev/s | p95  6.03 ms | drift  0.23 | probabilidad media 0.483


ventana @  200 eventos |  69 en ventana | throughput    69.7 ev/s | p95  6.15 ms | drift  0.23 | probabilidad media 0.360


ventana @  300 eventos |  70 en ventana | throughput    70.1 ev/s | p95  5.95 ms | drift  1.37 | probabilidad media 0.398  ← ALERTA de drift


ventana @  400 eventos |  66 en ventana | throughput    66.9 ev/s | p95  6.80 ms | drift  1.36 | probabilidad media 0.513
